#Pandas

##Uploading csv file

In [ ]:
#use ctrl+f9 to run all the sections everytime u enter this colab notenook

import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/pandas/matches.csv")
type(data)
data.head()

#Data manipulation (Fetching conditional rows) or masking

In [ ]:
#suppose i need to fetch 'kolkata' from city column and print it
fetch = data['city'] == 'Kolkata'
data[fetch]

##building a function for masking

In [ ]:
#to fetch specific location
def get_city(x):
  y = data['city']==x
  return data[y]

#to fecth shape of the row
def get_shape(x):
  y = data['city']=='x'
  return data[y].shape

In [ ]:
get_shape('Kolkata')

In [ ]:
get_city('Indore')

##Multiple Condition

In [ ]:
#we want natches played in kolkata after and on 2012 dataset
mask1 = data['city']=='Indore'
mask2 = data['date']>='2012-01-01'
data[mask1 & mask2] #here we used '&' instead of 'and' because mask1 and mask2 are boolean series
                    #type type(mask1) to show that it is a 'series'


##To calculate the count value of a catagorical data

In [ ]:
#most man of the match by individual
data['venue'].value_counts()


##Plotting  with pandas

In [ ]:
#pie chart
import matplotlib.pyplot as plt
data['winner'].value_counts().plot(kind='pie')

In [ ]:
data['winner'].value_counts().plot(kind ='bar')

In [ ]:
#ploting bar chart horizontally
data['winner'].value_counts().plot(kind ='barh')

In [ ]:
data['winner'].value_counts().tail().plot(kind='bar') #plotting last 5 data

In [ ]:
#Histogram
data['win_by_wickets'].plot(kind='hist')

##Series operation

In [ ]:
data['winner'].value_counts().index # to show the indexes

In [ ]:
data['winner'].value_counts().values #to show the count value

In [ ]:
winner = data['winner'].value_counts()
winner['Kolkata Knight Riders'] #it will show the count value

In [ ]:
 #to figure out most matches played by a team
y=data['team2'].value_counts()
x=data['team1'].value_counts()
z = x+y
z.nlargest(5)

##Sorting values

In [ ]:
#doing the previous example using sort_values()
y=data['team2'].value_counts()
x=data['team1'].value_counts()
z = x+y
z.sort_values(ascending=False).head()


In [ ]:
#or
(data['team1'].value_counts()+data['team2'].value_counts()).sort_values(ascending=False).head()

In [ ]:
#i want to sort on the basis of a particular col
data.sort_values('season',ascending=False)

In [ ]:
#i want to sort data based on multuple col and different ascending order
#fro multiple value sorting in python, always remember to assign an list
#here,first 'season' gets importance in order arrangement,then 'city' ,then 'win_by_runs'
data.sort_values(['season','city','win_by_runs'],ascending=[True,False,False])

In [ ]:
f = data['team1']=='Kolkata Knight Riders'
data[f]


##drop_duplicates()

In [ ]:
#it is used for deleting duplicates value keeping only unique value of a particular column
#the last match  won by any team  is te winner team of the IPL that year

s = data.drop_duplicates('season',keep = 'last').sort_values('season')[['season','winner']] #winners of IPL in every year
tw=s['season']>=2012 #winners of IPL after 2011
s[tw]

##Series to DataFrame coversion and Vice Versa

In [ ]:
#Target: to get most successfull team in IPL

df = data['team1'].value_counts() + data['team2'].value_counts()
col = df.rename_axis('Teams').reset_index(name= 'Match played')#For renaming 'index' of series to 'Teams' and
                                                               #Column 'Count' to 'Match played'
s = pd.DataFrame(col)      #series to dataframe conversion (pd.Series() used for dataframe to series conversion)
TMP = s.sort_values('Teams')      #we converted series to DF so that we cn perform sorting on the basis of 'Teams'
                                  #sort_values() only works for Dataframe
TMP

In [ ]:
#same as upper code comments to figure out most match wins
df = data['winner'].value_counts()
win = df.rename_axis('Teams').reset_index(name='Won')
win_matches = win.sort_values('Teams')
win_matches

In [ ]:
new = pd.merge(TMP,win_matches,on='Teams')
new['Lost'] = new['Match played'] - new['Won']
new['Success Ratio'] = new['Won']/new['Match played']
new.sort_values('Success Ratio',ascending = False)
data.head()

##groupby funciton

In [ ]:
data.head()
#used to make groups according to catagorical multiple row consist columns
ssn = data.groupby('season')
#to show the values of gruops,
ssn.size()

ssn.size().sort_values(ascending=False)  #used to sort the values

#to fetch the first value of each group as a dataframe
ssn.first()
#to fetch the first value of each group as a dataframe
ssn.last()

In [ ]:
#to know consist of groups and their indexes
ssn.groups
#to fetch any group
ssn.get_group(2011)
#To save time we can aslo do this : data.groupby('city').get_group('Kolkata')

##Arithmetic operation in groupby fn

In [ ]:
#sum
ssn.sum(['win_by_runs','win_by_wickets'])
#mean
ssn.mean(['win_by_runs','win_by_wickets'])
#max value
ssn.max(['win_by_runs','win_by_wickets'])

##Example using groupby

In [ ]:
import pandas as pd
dl = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/pandas/deliveries.csv")
dl.head()

In [ ]:
#find the top 5 most run scorer as a batsman
bat = dl.groupby('batsman')
most_runs = bat.sum('batsman_runs').sort_values('batsman_runs',ascending=False)
most_runs['batsman_runs'].head()

In [ ]:
#Finding the top 5 player with most sixes
y = dl['batsman_runs']==6
six = dl[y]

#Method 1
most_sixes_1 = six.groupby('batsman')['batsman_runs'].count().sort_values(ascending=False)
most_sixes_1.head()

#Method 2
most_sixes_2 = six.groupby('batsman').size().sort_values(ascending=False)
most_sixes_2.head()

In [ ]:
#most runs scored by V kohli againts an ipl team till 2017
dl.head()
y = dl['batsman']=='V Kohli'
vk = dl[y]
mr = vk.groupby('bowling_team').sum('batsman_runs')['batsman_runs'].sort_values(ascending=False).head(1)
mr

In [ ]:
#Creating a funcion
def most_runs_againts_team(x):
  y = dl['batsman']==x
  vk = dl[y]
  mr = vk.groupby('bowling_team').sum('batsman_runs')['batsman_runs'].sort_values(ascending=False).head(1)
  return mr

In [ ]:
most_runs_againts_team('MS Dhoni')

##isin() funciton

In [ ]:
from typing import AsyncContextManager
#find most strike rate of players in death overs(16-20overs) and
#who has played atleast 200 balls in death overs using isin() fn
#Strike rate = (total runs/total balls played)*100
dl
ov = dl[dl['over']>=16]              #data where over is equal or more then 16(death overs)
bat = ov.groupby('batsman').count()  #To count the balls total(no. of rows = every single ball)
x= bat['ball']>200                   #condition for a player player more than 200 balls
bat[x]                               #data manipulation
ab = bat[x].index.tolist()           #conerting the masking index into a list
final = ov[ov['batsman'].isin(ab)]   #filtered dataset and filtered by isin() fn
balls = final.groupby('batsman')['ball'].count()  #as count() function means total balls within 16-20 overs by each player
runs = final.groupby('batsman')['batsman_runs'].sum() #total runs within 16-20 overs by each player
SR = runs/balls*100
SR.reset_index(name='Srtike Rate')


##merge multiple dataset or dataframe

In [ ]:
#To figure out the most run scorer per season
#the 'data' dataset only consists the season but not players individual runs
#the 'dl' dataset do not contain any season column

merge_df = dl.merge(data,left_on = 'match_id',right_on='id')
oc = merge_df.groupby(['season','batsman'])['batsman_runs'].sum()
final = oc.reset_index().sort_values('batsman_runs',ascending=False).drop_duplicates('season',keep='first')
final.sort_values('season')[['season','batsman']]

##Pivot table

In [ ]:
#it is used for quick data fetching
#in this table, row consists 'seasons' and columns consists 'batsman'
merge_df.head()
ac = merge_df.pivot_table(index='season',columns='batsman',values = 'batsman_runs',aggfunc='sum')
#find active ipl years of any particular player
ac[ac['A Kumble']>0]['A Kumble']

##Correlation

In [ ]:
data.corr(numeric_only=True) #bcause corr() only works with numerical data

##Rename columns

In [ ]:
data.rename(columns={'id':'no. 0f balls','city':'place'})

##Set and reset index

In [ ]:
#set
data.set_index('id').head()
#or another example
set_data = data[['id','winner']].rename(columns={'id':'match'}).set_index('match').head()
set_data

In [ ]:
#Reset
set_data.reset_index()
#u want to convert index into a named column
set_data.reset_index().rename_axis('index')

##dropna() function

In [ ]:
#to find out f any coumn has null value or not
dl.isnull().any() #any column with null value is 'True'
dl.isnull().sum() #to find out total missing value per column

In [ ]:
#eliminating rows of empty values
print(dl.shape)
print(dl.dropna(axis=0).shape) #it will deop any row containing single or multiple NaN value or null value

print(dl.dropna(axis=1).shape) #it has 3 columns with NaN values
                               #(any single value of 'NaN',this command will drop that column)

print(dl.dropna(axis=0,how='all').shape) #any row with all its value 'Nan' is removed
                                         #here in this dataset,there is no row with all its value NaN

print(dl.dropna(axis=1,how='all').shape) #any column with all its value 'Nan'
                                         #here in this dataset,there is no col with all its value NaN

dl.dropna(axis=0,subset = ['player_dismissed','fielder']).shape
                                         #it removes particular these two columns 'NaN' valued rows

##fillna() function

In [ ]:
from types import MethodType
#use this function for each column which has missing values
#to find the columns with missing values, use isnull().any()
#one line of code at a time

data.isnull().any()
data.isnull().sum()
data['city'].fillna('Somewhere')
data.isnull().any()

#using method to fill out the missiong values
dl['player_dismissed'].fillna(method='ffill') #to filling missing values with upper placed value
dl['player_dismissed'].fillna(method='bfill') #to filling missing values with lower placed value

#another method for using ffill and bfill
dl['player_dismissed'].ffill()
dl['player_dismissed'].bfill()